# Python cheatsheet: Excel extraction for NSSF Tier II (annotated)

One notebook for everything you have been doing, in three layers:

1. **The base code**, explained line by line: first for the **master files**, then for the **contribution files** (starter check, extraction, settings, checking code).
2. **What differed from the base**, as short entries, with no names. Each entry says **what happened**, **how it was fixed**, shows the code, explains it line by line, and lists other options.
3. **A table of how the base grew**: which line was added, and which problem it solved.

**How the comments are written**
- `WHAT:` what the line does.
- `WHY:` the reason it is there.
- `OR:` another way to get the same result.
- `WATCH:` something that can go wrong.

**Do not use Run All on this cheatsheet.** The file paths are placeholders, and the fragments in section 4 are written in text blocks so that they cannot run by accident. Copy a fragment into the base code where the entry says. In a working notebook with your real paths, restart the kernel and Run All is a good habit (see 4.7.3). For a ready-to-run version of the base code that is driven by settings, use `Master_Extraction_Notebook.ipynb`.

### Where to look when something breaks

| What you see | Entry |
|---|---|
| `ModuleNotFoundError`, or an error mentioning `xlrd` or `openpyxl` | 1, 4.1.1 |
| `KeyError: 'Some heading'` | 4.2.1, 4.2.2, 4.2.5 |
| Columns come out wrong or shifted, or names like `Unnamed: 3` | 4.1.2, 4.1.3, 4.2.3 |
| A Pers NO lost its leading zeros | 4.2.4 |
| The total differs from Excel | 4.3.1, 4.3.2, 4.4.1, 4.4.2 |
| A total like `229686.21000000002` | 4.4.3 |
| A date appears as `2026-03-01 00:00:00` | 4.4.4 |
| `NameError`, `TypeError`, `FileNotFoundError`, `Worksheet not found`, `IndentationError`, or `5 columns passed, data had 6` | 4.7.1 |
| The file or folder cannot be found, or you do not know where the txt was saved | 4.7.2 |
| The checker says "expected N total rows" | 4.1.5, 4.5.2 |
| The checker marks every row of a column as different | 4.4.1, 4.4.4, 4.5.2 |

### Contents
1. pip
2. Base code: master files
3. Base code: contribution files
4. What differed from the base, and how it was fixed
5. How the base grew

## 1. pip: Python's installer

pip downloads libraries (pandas, openpyxl, nbformat) from PyPI, the public online library of Python packages, and installs them so `import` works.

- `import x` only loads a library that is already installed.
- `ModuleNotFoundError` means the library is not installed in the Python this notebook uses.
- In a notebook, use `%pip`, not `pip`. `%pip` installs into the same Python the notebook runs on.
- The notice "A new release of pip is available" is only a reminder, not an error. Ignore it or update pip.

In [ ]:
# Install a library. Remove the # to run it. After installing, restart the kernel.
# %pip install nbformat

# Replace the installed version with the newest one. --upgrade makes it replace instead of skip.
# %pip install --upgrade pandas

# Remove a library. -y answers "yes" to the confirmation question automatically.
# %pip uninstall -y nbformat

# List every installed library with its version number.
%pip list

# Show details for one library: version, location, and what it depends on.
%pip show pandas

# Update pip itself (run in a terminal, not in a notebook cell).
# python.exe -m pip install --upgrade pip

## 2. Base code: master files

A master file holds employee details (names, ID, KRA PIN, membership number). It has **no amounts and no totals**, so there is no loop: read the sheet, keep eight columns, write the txt.

In [ ]:
import pandas as pd
# WHAT: loads pandas, the library for tables (DataFrames). "as pd" gives it the short nickname pd.

df = pd.read_excel(r"path/to/master_file.xlsx", dtype={"Pers NO": str})
# WHAT: reads the Excel file into a table called df. Here df really is a table (the data).
# WHY no sheet_name: only the FIRST sheet is read (the default is sheet_name=0). To read another sheet, add
#      sheet_name="SHEET NAME", and list the names first with pd.ExcelFile(path).sheet_names.
# WHY dtype={"Pers NO": str}: keeps Pers NO as text, so a number such as 007123 does not become 7123.
# OR: dtype=str makes every column text.
# WATCH: reading an old .xls file needs xlrd (see entry 4.1.1).

df_selected = df[['Payroll Area','Factory code','Pers NO','Surname','Other Name(s)','ID','KRA PIN','Membership No']]
# WHAT: keeps eight columns, in the order of the list. The inner [...] is the list of names; the outer [...] picks columns.
# WATCH: every name must match the Excel heading exactly (capitals, spaces and full stops). One wrong name gives a KeyError.
#        The headings are different in different files: see entries 4.2.1 and 4.2.2.

df_selected.to_csv("FactoryMaster.txt", sep="|", index=False, header=False)
# WHAT: writes the table to a text file.
#   sep="|"        separate the columns with | instead of a comma
#   index=False    do not write pandas' own row numbers
#   header=False   do not write the column names as the first line
# OR: na_rep="NULL" writes every missing value as the word NULL (see entry 4.5.2).

print(df.columns.tolist())
# WHAT: prints the real headings as a list, so hidden spaces are visible.
# WHY at the end: if the selection above fails with a KeyError, run this line alone first and copy the exact names.

## 3. Base code: contribution files

The pattern for every contribution file:
1. **Starter check:** open the Excel file and print each sheet's name and column headings.
2. **Extraction:** keep six columns, add a total row after each month, add a grand total at the end, and write a `|`-separated txt file.
3. **Settings:** the names the checking code needs.
4. **Checking code:** read the txt file back and compare it with the Excel file.

In [ ]:
import pandas as pd   # WHAT: loads pandas, the library for working with tables. "as pd" gives it the short nickname pd.
                      # WHY: read_excel, DataFrame and concat all come from pandas.
import numpy as np    # WHAT: loads numpy, the library for numerical work.
                      # WHY: it is used later for np.nan, the standard "missing value".

df = r"path/to/your/file.xlsx"   # check the extension is not doubled (.xlsx.xlsx)
# WHAT: stores the file's location as text.
# WATCH: despite the name, df holds only the path (a string), not a table. A table is also usually called df,
#        which is why "df is not defined" errors are easy to cause. A clearer name would be file_path.
# WHY the r before the quotes: it makes a "raw string", so Windows backslashes (C:\Users\...) are not read as
#        special characters such as \n (new line) or \t (tab).
# OR: forward slashes ("C:/Users/..."), or pathlib: from pathlib import Path; Path("C:/Users/.../file.xlsx").

all_sheets = pd.read_excel(df, sheet_name=None, dtype={"Pers NO": str})
# WHAT: reads the Excel file. sheet_name=None means "read every sheet". The result is a dictionary:
#       {"NOV": table, "DEC": table, ...}, one table per sheet.
# WHY dtype={"Pers NO": str}: it forces the Pers NO column to stay as text. As a number, 007123 would become 7123
#       and the leading zeros would be lost.
# OR sheet_name: 0 = first sheet only, "NOV" = one sheet by name, ["NOV", "DEC"] = a chosen few.
# OR dtype: dtype=str makes every column text; converters={"Pers NO": str} works per column.
# OR to see the sheet names without loading the data: pd.ExcelFile(df).sheet_names
# WATCH: reading a legacy .xls file needs the xlrd library (%pip install xlrd).

# Starter check for every new notebook: sheet names and the headings of each sheet
for sheet_name, sheet_df in all_sheets.items():
    # WHAT: loops through the dictionary. .items() gives (name, table) pairs, one pair per loop.
    # sheet_name is the tab's name (for example "NOV"); sheet_df is that tab's table.
    # OR: for name in all_sheets: (names only), then all_sheets[name] to get the table.
    print(sheet_name, sheet_df.columns.tolist())
    # WHAT: prints the tab name and its column headings as a list.
    # .columns is the heading row; .tolist() turns it into a normal Python list so the spaces are visible.
    # WHY: a trailing space ("Payroll Area ") or a double space ("Payroll  Month") is invisible in Excel,
    #      but Python needs the exact text.
    # OR: print(sheet_df.columns) shows the same with extra detail; sheet_df.head() shows the first rows too.

### 3.2 Extraction code

**What it produces:** a txt file where each month's rows are followed by a month total row, and the whole file ends with a grand total row.

**The steps inside the loop, for every sheet:**
1. Keep only the six columns you need.
2. Keep only rows that have a Pers NO (this drops blank rows and any total row already inside the sheet).
3. Turn blank cells into proper missing values.
4. Convert Tier2 Cont to numbers.
5. Add the month's Tier2 Cont total.
6. Stack the rows and a total row onto the combined table.

After the loop: add the grand total row and write the file.

In [ ]:
import pandas as pd
import numpy as np
# WHAT: loads pandas and numpy again. Each cell can run on its own after a kernel restart, so repeating the imports is harmless.
# WATCH: the numpy line was added in this cheatsheet. The blank-to-NaN line below uses np.nan. In your own notebooks this cell imported only pandas, and it worked only because the starter check had already run import numpy. After a kernel restart, running just this cell gave: NameError: name 'np' is not defined.

all_sheets = pd.read_excel(
    r"path/to/your/file.xlsx",  # check the extension is not doubled (.xlsx.xlsx)
    sheet_name=None,             # WHAT: read every sheet into a dictionary of tables
    dtype={"Pers NO": str}       # WHAT: keep Pers NO as text so leading zeros survive
)
# WHAT: same call as in the starter check, written across several lines so each setting has its own line.
# WATCH: the file path here must be edited for every Excel file, and so must the .txt path at the end.

combined = pd.DataFrame()
# WHAT: creates an empty table. It will collect every month's rows and totals, and becomes the file you export.
# OR (faster for big data): collect each piece in a list (pieces = []), then run pd.concat(pieces) once at the end.
#     Calling concat inside the loop copies the whole table every time, which is slower, but fine for your sizes.

month_totals = []
# WHAT: an empty list. Each month's Tier2 Cont total is added to it, so the grand total can be worked out later.

columns = [
    "Payroll Area ",
    "FACTORY CODE (Station)",
    "Pers NO",
    "Voluntary Amount",
    "Tier2 Cont",
    "Payroll  Month"
]
# WHAT: the six columns to keep, in the order they must appear in the txt file.
# WATCH: the names must match the Excel headings exactly. "Payroll Area " ends with a space and "Payroll  Month"
#        has two spaces. A mismatch causes a KeyError. The starter check above shows the exact text.
# OR: clean headings first with sheet_df.columns = sheet_df.columns.str.strip(), but then this list must match
#     the cleaned names.

for sheet_name, sheet_df in all_sheets.items():
    # WHAT: loops through every sheet, one at a time.

    selected = sheet_df[columns].copy()
    # WHAT: keeps only the six columns, in the order of the list.
    # WHY .copy(): it makes an independent table, so the changes below do not touch the original sheet and do not
    #      trigger pandas' SettingWithCopyWarning.

    selected = selected[selected["Pers NO"].notna() & (selected["Pers NO"].astype(str).str.strip() != "")]
    # WHAT: keeps only rows that have a Pers NO. The part inside [ ] is a True/False test for every row.
    #   .notna()              True when the cell is not empty
    #   .astype(str).str.strip() != ""   True when the cell is not just spaces
    #   &                     "and": both tests must be True. Each test needs its own brackets.
    # WHY: removes blank rows and any total row already sitting in the sheet, so nothing is counted twice.
    # OR: selected.dropna(subset=["Pers NO"]) is shorter, but it keeps cells that contain only spaces.
    # OR: selected.query('`Pers NO`.notna()') (the backticks are needed because of the space in the name).

    selected = selected.replace(r"^\s*$", np.nan, regex=True)
    # WHAT: turns every cell that is empty or only spaces into NaN (the standard missing value).
    #   ^ start of text, \s* any number of spaces, $ end of text. regex=True makes pandas read it as a pattern.
    # WHY: so every blank is the same kind of missing value, and na_rep="NULL" at the end writes them all as NULL.
    # OR: selected.fillna("") if you want blanks written as empty text instead of NULL.

    selected["Tier2 Cont"] = pd.to_numeric(selected["Tier2 Cont"], errors="coerce").fillna(0)
    # WHAT: converts the Tier2 Cont column to numbers. errors="coerce" turns anything that cannot be a number
    #       into NaN, and .fillna(0) then replaces every NaN with 0.
    # WHY: so the sum works even when a few cells were stored as text.
    # OR: errors="raise" stops with an error on the first bad value, which is useful to find bad data.
    # WATCH: fillna(0) hides problems silently. To see what was changed, run before the conversion:
    #        bad = selected[pd.to_numeric(selected["Tier2 Cont"], errors="coerce").isna()]

    this_month_total = selected["Tier2 Cont"].sum()
    # WHAT: adds up every Tier2 Cont value for this one sheet.
    # WATCH: sums of decimals can show tiny float errors such as 229686.21000000002. round(..., 2) tidies it.

    month_totals.append(this_month_total)
    # WHAT: adds this month's total to the end of the list.

    total_row = pd.DataFrame(
        [["", "", "", "", this_month_total, ""]],
        columns=columns
    )
    # WHAT: builds a one-row table for the total line. It has blanks everywhere except Tier2 Cont.
    # The outer [ ] is the list of rows and the inner [ ] is the one row. The values follow the order of
    # `columns`, so this_month_total sits in the fifth position, which is Tier2 Cont.
    # WATCH: if you change the order of `columns`, change the position of the total here as well.

    combined = pd.concat([combined, selected, total_row], ignore_index=True)
    # WHAT: stacks, in this order, what was already collected, this month's rows, and the total row.
    # ignore_index=True renumbers the rows 0, 1, 2, 3... instead of restarting from 0 for every sheet.

grand_total =sum(month_totals)
# WHAT: adds up the list of monthly totals (Python's built-in sum, not the pandas one).
# WHY from the list: if you summed the Tier2 Cont column of `combined`, the month total rows inside it would be
#      counted too, and the answer would be too high.
# OR: np.sum(month_totals), or sum([...]) with round(..., 2) around it.

grand_total_row = pd.DataFrame(
    [["", "", "", "", grand_total, ""]],
    columns=columns
)
# WHAT: the same one-row layout as total_row, holding the grand total.

combined = pd.concat([combined, grand_total_row], ignore_index=True)
# WHAT: adds the grand total row to the very end.

combined.to_csv(
    "path/to/output/FactoryContribution.txt",  # specify your desired output path and filename
    sep="|",          # WHAT: separate columns with | instead of a comma, because the upload format needs it
    index=False,      # WHAT: do not write pandas' own row numbers
    header=False,     # WHAT: do not write the column names as the first line
    na_rep="NULL"     # WHAT: write every missing value as the word NULL
)
# WHAT: writes the table to a text file.
# WATCH: na_rep only affects true missing values (NaN). The "" cells in the total rows stay empty, which is why
#        the checking code below treats both "" and "NULL" as "no Pers NO".
# OR: float_format="%.2f" forces two decimals on every number. encoding="utf-8" fixes strange symbols.
# OR: .to_csv() with a different extension, such as .csv, if the file does not have to be .txt.

### 3.3 Settings for the Excel vs TXT check

These are the only lines to edit for each factory before running the checking code. Each name must match your real file.

In [ ]:
amount_col = "Tier2 Cont"
# WHAT: the name of the column that holds the money amounts. The checker adds up this column.

voluntary_col = "Voluntary Amount"
# WHAT: the second money column. It is compared as a number too, so 100 and 100.00 count as equal.

id_col = "Pers NO."
# WHAT: the column that identifies an employee row. Rows with a value here are employees; rows without one are totals.
# WATCH: this must match the heading in THIS Excel file exactly. Some files use "Pers NO", others "Pers NO."
#        with a full stop. Edit it per file, and keep it the same as the entry in the `columns` list.

fill_empty = "NULL"
# WHAT: the word that the txt file uses for missing values. It must equal the na_rep used in the extraction.

output_file = "FactoryContribution.txt"   # the exact name your export wrote
# WHAT: the txt file to check. A bare name means "in the same folder as this notebook".

skip_sheets = []
# WHAT: a list of sheet names to leave out of the comparison, for example ["Summary"]. Empty means "check all".

add_grand_total = True
# WHAT: True means the file ends with a grand total row. Set it to False if your extraction did not add one.

def clean_amount(s):
    # WHAT: defines a small reusable tool. s is whatever column you hand it.
    return pd.to_numeric(s, errors="coerce").fillna(0)
    # WHAT: turns text into numbers and blanks or bad values into 0, the same rule as in the extraction.
    # OR: clean_amount = lambda s: pd.to_numeric(s, errors="coerce").fillna(0)   (same thing in one line)

### 3.4 Checking code

**The idea:** the extraction writes a txt file. This code reads it back and checks, sheet by sheet, that
1. the row counts are the same,
2. each month's total in the txt equals the sum of the employee rows in the Excel,
3. every column matches, value by value,
4. the grand total is right.

The result is a comparison table with one line per sheet.

In [ ]:
def tidy(s):
    # WHAT: a helper that makes text comparable. s is a column.
    s = s.where(s.notna(), "" if fill_empty is None else fill_empty).astype(str)
    # WHAT: .where(test, other) keeps a value when the test is True and uses `other` when it is False.
    #       So every missing value becomes fill_empty ("NULL"), or "" if fill_empty is None. Then everything becomes text.
    # WHY: so a missing value in Excel and the word NULL in the txt are treated as the same thing.
    return s.str.replace("\xa0", "", regex=False).str.strip()
    # WHAT: removes non-breaking spaces (\xa0, an invisible space that Excel sometimes holds), then trims spaces at the ends.
    # regex=False means "treat \xa0 as plain text, not as a pattern".

def to_num(s):
    # WHAT: a helper that turns a column into numbers even when it holds symbols or text.
    return pd.to_numeric(tidy(s).str.replace(r"[^\d.-]", "", regex=True), errors="coerce")
    # WHAT: tidy first, then delete everything that is not a digit, a full stop, or a minus sign
    #       ([^...] means "any character except these"), then convert to numbers. Bad values become NaN.
    # WHY: so "1,200.50" or " 1200.5" and 1200.5 compare as equal.

number_cols = [c for c in [amount_col, voluntary_col] if c is not None and c in columns]
# WHAT: a list comprehension: build a list by looping. It keeps the money columns that are set and that exist in `columns`.
# WHY: these columns are compared as numbers; all the others are compared as text.
# OR: the same with a normal for loop and an if statement, spread over several lines.

out = pd.read_csv(output_file, sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
# WHAT: reads the txt file back into a table.
#   sep="|"                 the columns are separated by |
#   header=None             the file has no heading line
#   names=columns           use the six column names from the list
#   dtype=str               read everything as text, so nothing is changed on the way in
#   keep_default_na=False   do not turn the text "NULL" or blanks into missing values; keep them as written
# WHY: the check must see exactly what is in the file.

is_employee = ~out[id_col].str.strip().isin(["", "NULL"])
# WHAT: a True/False value for every line of the file. isin([...]) is True when the Pers NO is blank or NULL.
#       The ~ flips it, so True means "this line has a Pers NO, so it is an employee row".

is_total = ~is_employee & (out[amount_col].str.strip() != "")
# WHAT: True for lines that are not employee rows but do have an amount. These are the total rows.

block = is_total.cumsum()
# WHAT: a running count of total rows seen so far (True counts as 1). cumsum = cumulative sum.
# WHY: every employee row of the first sheet has block 0, the second sheet block 1, and so on,
#      so the file can be split back into sheets.

file_totals = to_num(out.loc[is_total, amount_col]).round(2).tolist()
# WHAT: takes the amounts on the total rows, converts them to numbers, rounds them to 2 decimals, and makes a list.
# .loc[rows, column] picks rows by a True/False test and one column by name.

sheets_used = [(n, d) for n, d in all_sheets.items() if n not in skip_sheets]
# WHAT: a list of (name, table) pairs for the sheets to check, leaving out any in skip_sheets.

expected_totals = len(sheets_used) + (1 if add_grand_total else 0)
# WHAT: how many total rows the file should hold: one per sheet, plus one more for the grand total if there is one.
# `1 if add_grand_total else 0` is a one-line if/else.

totals_ok = len(file_totals) == expected_totals
# WHAT: True when the file holds exactly the number of total rows we expected.
if not totals_ok:
    print("WARNING: expected", expected_totals, "total rows in the txt file, found", len(file_totals))
    # WHAT: warns you when the counts differ. Then the per-sheet total comparison is skipped (see below).

results = []
# WHAT: an empty list that will hold one summary line (a dictionary) per sheet.
for i, (name, df) in enumerate(sheets_used):
    # WHAT: loops over the sheets. enumerate adds a counter i (0, 1, 2...) next to each (name, table) pair.
    # WATCH: here df really is a table (the sheet), unlike the df in the starter check, which was a path.

    amt = clean_amount(df[amount_col])
    # WHAT: the Tier2 Cont column of this sheet, as numbers.

    has_id = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    # WHAT: True for the rows in the Excel that have a Pers NO (the employee rows).

    exp = df.loc[has_id, columns].reset_index(drop=True)
    # WHAT: the "expected" table: only the employee rows and the six columns, with rows renumbered from 0.

    got = out[is_employee & (block == i)][columns].reset_index(drop=True)
    # WHAT: the "got" table: the employee rows in the txt file that belong to this sheet (block == i).

    employee_sum = round(amt[has_id].sum(), 2)
    # WHAT: the total of the employee rows in Excel, rounded to 2 decimals.
    own_total = round(amt[~has_id].sum(), 2)
    # WHAT: the total of the rows WITHOUT a Pers NO, which is a total row that already exists in the sheet (or 0 if none).

    row = {"Sheet": name, "Rows in Excel": len(exp), "Rows in txt": len(got),
           "Employee sum": employee_sum,
           "Sheet's own total row": own_total if own_total != 0 else None}
    # WHAT: a dictionary (name: value pairs) that will become one line of the final table.

    if totals_ok:
        row["Total in txt"] = file_totals[i]
        row["Totals match"] = abs(employee_sum - file_totals[i]) < 0.005
        # WHAT: compares the txt total for this sheet with the Excel sum.
        # WHY not ==: decimal numbers can differ by tiny amounts. Treating anything under half a cent as equal is safe.
    else:
        row["Total in txt"] = None
        row["Totals match"] = None

    row["Matches sheet's own total"] = "no total row" if own_total == 0 else bool(abs(employee_sum - own_total) < 0.005)
    # WHAT: checks whether the employee rows add up to the total row printed in the Excel sheet, if there is one.

    if len(exp) != len(got):
        row["Columns that differ"] = "row counts differ"
        # WHAT: if the row counts differ, comparing column by column makes no sense, so say so.
    else:
        bad = []
        # WHAT: collects the names of the columns that do not match.
        for col in columns:
            if col in number_cols:
                a, b = to_num(exp[col]), to_num(got[col])
                same = (a.isna() & b.isna()) | ((a - b).abs() < 0.005)
                # WHAT: money columns: equal when both are missing, or when they differ by less than half a cent.
            else:
                same = tidy(exp[col]) == tidy(got[col])
                # WHAT: other columns: equal when the tidied text is identical.
            if not same.all():
                bad.append(col + " (" + str(int((~same).sum())) + " rows)")
                # WHAT: if any row differs, record the column name and how many rows differ. ~same flips True/False.
        row["Columns that differ"] = ", ".join(bad) if bad else "none"
        # WHAT: joins the bad column names with commas, or writes "none" if every column matched.
    results.append(row)
    # WHAT: adds this sheet's summary line to the list.

compare = pd.DataFrame(results)
# WHAT: turns the list of dictionaries into a table, one line per sheet.

if add_grand_total and totals_ok:
    print("Grand total correct:", abs(file_totals[-1] - compare["Employee sum"].sum()) < 0.005)
    # WHAT: file_totals[-1] is the last total in the file, the grand total. It is compared with the sum of all sheets.
print("Sheets identical to the txt:", int((compare["Columns that differ"] == "none").sum()), "of", len(compare))
# WHAT: counts the sheets whose "Columns that differ" says "none", and prints it out of the total number of sheets.

compare
# WHAT: shows the comparison table. A notebook displays the last expression in a cell automatically.
# OR: display(compare) from IPython.display shows it from anywhere in the cell.
# A clean result: every "Totals match" is True and every "Columns that differ" is "none".

## 4. What differed from the base, and how it was fixed

Each entry is a **fragment**. It says where it goes in the base code. The fragments are in text blocks so you can copy them but they will not run by accident.

---
### 4.1 Reading the file

#### 4.1.1 An old `.xls` file, or a missing library

**What happened:** reading the file stopped with an error that mentions `xlrd` (old `.xls` files) or `openpyxl` (newer `.xlsx` files), or `import` failed with `ModuleNotFoundError`.

**How it was fixed:** install the library into the same Python the notebook uses, then restart the kernel.

```python
%pip install xlrd              # reads old .xls files
%pip install openpyxl          # reads .xlsx files
%pip install pandas xlrd       # several libraries in one line
```

- `%pip` is a notebook command that installs into the Python the notebook is running on. A plain `pip` in a terminal may install into a different Python.
- After installing, use **Kernel, then Restart**. Python only sees new libraries after a restart.
- **WATCH:** `pip` needs internet. See section 1 for what pip is.

---
#### 4.1.2 A sheet with no heading row

**What happened:** one sheet had no usable heading row, so pandas used the first data row as headings and the names were wrong.

**How it was fixed:** read that sheet again with `header=None`, and give the headings by hand with `names=`. Put the result back into the `all_sheets` dictionary, so the rest of the code works unchanged.

```python
sheet_names = [                                   # all headings, in left-to-right order of the sheet
    "e", "FACTORY CODE (Station)", "Prev Pers NO", "Pers NO", "Full Name",
    "ID", "KRA PIN", "Membership No", "Gross Amount Tier1", "Earnings",
    "Standard Amount", "Voluntary Amount", "Total Amount", "Tier1 Cont",
    "Tier2 Cont", "Payroll  Month"
]

all_sheets = pd.read_excel(path, sheet_name=None, dtype={"Pers NO": str})     # normal read of every sheet
first = list(all_sheets.keys())[0]                                            # name of the first sheet
all_sheets[first] = pd.read_excel(path, sheet_name=first, header=None,
                                  names=sheet_names, dtype={"Pers NO": str})  # read it again, with our headings
```

- `header=None` means "there is no heading row, every row is data". Without it, the first data row would be used as the headings.
- `names=sheet_names` gives the column names. **WATCH:** the list must have exactly as many names as the sheet has columns, or pandas raises an error. The first name does not have to be the real heading. In the `columns` list you use the same text.
- `list(all_sheets.keys())[0]` is the first sheet's name: `.keys()` lists the names, `list(...)` makes a normal list, and `[0]` takes the first. **OR:** `pd.ExcelFile(path).sheet_names[0]`.
- `all_sheets[first] = ...` replaces that sheet's table inside the dictionary. The sheet keeps its place in the order.
- **WATCH:** if the sheet did have a heading row, it is now a data row. Check with `.head()`.

---
#### 4.1.3 Headings start lower down in the sheet

**What happened:** the headings were not on the first row, because rows above them held titles or blanks.

**How it was fixed:** `header=` says which row holds the headings, counting from 0. It can be used on one sheet, or on a master file.

```python
all_sheets["SHEET NAME"] = pd.read_excel(path, sheet_name="SHEET NAME", header=2, dtype={"Pers NO": str})
df = pd.read_excel(r"path/to/master_file.xlsx", header=1, dtype={"Pers NO": str})
```

- `header=2` means "the third row holds the headings, ignore the rows above it". `header=1` means the second row.
- The first line replaces one sheet in the dictionary; the second reads a master file whose headings are on the second row.
- **WATCH:** the sheet name must be typed exactly, with capitals and spaces. Print `all_sheets.keys()` or `pd.ExcelFile(path).sheet_names` to copy it. A trailing space in a sheet name is common: use `repr(name)` to see it.
- Check with `print(df.columns.tolist())` after reading.

---
#### 4.1.4 All the months are in one sheet

**What happened:** the Excel file had a single sheet with every month in it, instead of one sheet per month.

**How it was fixed:** read the one sheet, then split it into one table per month with `groupby`. The result is a dictionary again, so everything after it is unchanged.

```python
one_sheet = pd.read_excel(r"path/to/file.xlsx", dtype={"Pers NO": str})
all_sheets = {m: g for m, g in one_sheet.groupby("Payroll  Month", sort=False)}
```

- There is no `sheet_name=None`, because there is only one sheet to read.
- `one_sheet.groupby("Payroll  Month")` splits the rows into groups, one group for every different value in that column.
- Looping over a groupby gives pairs: `m` is the month value and `g` is that group's table.
- `{m: g for m, g in ...}` is a **dictionary comprehension**. It builds `{month: table}`, the same shape that `sheet_name=None` gives.
- `sort=False` keeps the groups in the order they first appear in the sheet. The default sorts them.
- **WATCH:** the groups are decided by the exact value, so a month typed differently in a few rows becomes a separate group. The names printed by the loop are now the month values, not sheet names.
- **WATCH:** the checking code must build `all_sheets` the same way, or it will compare against the wrong tables.

---
#### 4.1.5 Skip a sheet, or read only some

**What happened:** an extra sheet (for example a summary or a master-data sheet) was pulled in by `sheet_name=None`, or one sheet was not wanted.

**How it was fixed:** skip it inside the loop, and tell the checking code to skip it too.

```python
# Skip a sheet by name:
if sheet_name == "SHEET NAME":
    continue

# Skip the last sheet without typing its name:
last_sheet_name = list(all_sheets.keys())[-1]        # before the loop
for sheet_name, sheet_df in all_sheets.items():
    if sheet_name == last_sheet_name:
        continue                                      # skip this sheet entirely

# Process only the sheets that have the column:
if "Tier2 Cont" in sheet_df.columns:
    ...

# In the settings of the checking code:
skip_sheets = ["SHEET NAME"]
```

- `continue` jumps to the next sheet at once, so nothing after it runs for the skipped sheet.
- `[-1]` is the last item of a list. `[0]` is the first.
- `"Tier2 Cont" in sheet_df.columns` is `True` when the sheet has that heading.
- **WATCH:** if the extraction skips a sheet and `skip_sheets` does not, the checker warns "expected N total rows" and skips the per-sheet total check.
- **OR:** read only the sheets you want: `pd.read_excel(path, sheet_name=["JAN", "FEB"])` gives a dictionary of just those.

---
### 4.2 Headings

#### 4.2.1 Headings are spelled differently in different files

**What happened:** `KeyError: 'Some heading'`. The headings you had in `columns` did not match the Excel headings. The differences you met were:
- a trailing space (`"Payroll Area "` against `"Payroll Area"`),
- a double space against a single space (`"Payroll  Month"` against `"Payroll Month"`),
- different capitals (`"PAYROLL AREA"`, `"PayRoll  Month"`),
- a **leading** space (`" Tier2 Cont"`),
- different wording (`"FACTORY CODE (Station)"`, `"Factory code"`, `"FACTORY CODE"`),
- a misspelling in the file (`"Payrol area"`),
- a full stop (`"Pers NO."`).

**How it was fixed:** copy the heading text exactly, as the starter check prints it, into the `columns` list (and into every place that uses the name). To see the hidden characters:

```python
print(repr("Payroll Area "))                   # repr shows the quotation marks, so a trailing space is visible
print(sheet_df.columns.tolist())               # the headings of one sheet, as a list
pd.set_option("display.max_columns", None)     # show every column in a table, with no "..." in the middle
```

When the headings differ **between sheets of the same file**, compare them all at once with a grid:

```python
all_cols = []
for sheet_df in all_sheets.values():
    for c in sheet_df.columns:
        if c not in all_cols:
            all_cols.append(c)                 # every heading seen in any sheet, once, in order of appearance

grid = pd.DataFrame(
    {name: ["✔" if c in sheet_df.columns else "" for c in all_cols]
     for name, sheet_df in all_sheets.items()},
    index=all_cols
)
grid                                           # one row per heading, one column per sheet
```

- `all_cols` collects each different heading once. `if c not in all_cols` is what stops repeats. **OR:** `list(dict.fromkeys(...))` does the same in one line.
- The grid has one **row** per heading and one **column** per sheet. A tick (✔) means "this sheet has this heading". A row with a missing tick shows a sheet that is different.
- To check only your target columns against one sheet:

```python
last_sheet_name = list(all_sheets.keys())[-1]
last_cols = all_sheets[last_sheet_name].columns.tolist()
matches = [c for c in columns if c in last_cols]      # keep each target column that the sheet really has
print("Columns that matched:", matches)               # compare with `columns` to see which ones are missing
```

- **WATCH:** the headings must be changed everywhere the name is used: the `columns` list, `["Tier2 Cont"]` in the sum, the position of the total in `total_row`, and `amount_col` and `id_col` in the settings.

---
#### 4.2.2 Headings differ between sheets of the same file

**What happened:** the headings were not the same in every sheet (an extra space, a different capital, a different wording), so one list could not match them all.

**How it was fixed:** there were four ways, from the simplest to the most flexible.

**(a) Trim spaces from every heading before the loop.**
```python
for name in all_sheets:
    all_sheets[name].columns = all_sheets[name].columns.astype(str).str.strip()
```
- `for name in all_sheets` loops over the sheet names.
- `.columns.astype(str)` makes every heading text (a heading can be a number or a date). `.str.strip()` removes spaces at the start and end.
- Assigning to `.columns` renames them. Then `columns = [...]` must be written **without** the extra spaces.
- **WATCH:** it does not fix a double space in the middle, or a different capital.

**(b) Rename the odd heading to the standard one.**
```python
for sheet_name, sheet_df in all_sheets.items():
    sheet_df = sheet_df.rename(columns={"Payroll Month": "Payroll  Month"})
    selected = sheet_df[columns].copy()
```
- `.rename(columns={old: new})` changes the named headings only. Headings that are not in the dictionary stay as they are, and if none match, nothing happens.
- It returns a new table, so it is assigned back to `sheet_df`. The checking code needs the same line for its `df`.

**(c) Match headings after collapsing the spaces, and rename them to the standard list.**
```python
for sheet_name, sheet_df in all_sheets.items():
    normalized_columns = {" ".join(str(c).split()): c for c in sheet_df.columns}
    missing = [c for c in columns if " ".join(c.split()) not in normalized_columns]
    if missing:
        raise KeyError(f"{sheet_name}: missing columns {missing}")

    source_columns = [normalized_columns[" ".join(c.split())] for c in columns]
    selected = sheet_df[source_columns].copy()
    selected.columns = columns
```
- `str(c).split()` cuts a heading at every run of spaces. `" ".join(...)` glues the pieces back with **one** space. So `"Payroll  Month"` and `"Payroll Month"` both become `"Payroll Month"`.
- `normalized_columns` is a dictionary: cleaned heading → the real heading in the sheet.
- `missing` lists every wanted heading that has no match, even after cleaning. If any is missing, `raise KeyError(...)` stops with a message that names the sheet and the headings. `f"..."` is an f-string: `{...}` inside it is replaced by the value.
- `source_columns` is the list of **real** headings in the order of `columns`.
- `sheet_df[source_columns]` selects them, and `selected.columns = columns` renames them to the standard names. After that, the rest of the code works unchanged.
- **WATCH:** it fixes spacing only, not capitals or different words. The checking code needs the same lookup for the Excel side, or the two tables will not match: `exp = df.loc[has_id, source_cols].copy()` followed by `exp.columns = columns`.

**(d) Create a missing column as empty, instead of stopping.**
```python
selected_raw = sheet_df.reindex(columns=columns)
this_month_total = selected_raw["Tier2 Cont"].sum()
selected = selected_raw.fillna("Null")
```
- `.reindex(columns=columns)` returns a table with exactly those columns. A column the sheet does not have is **created and filled with NaN** instead of raising a `KeyError`.
- The total is taken from `selected_raw` **before** `fillna("Null")`, because after it the cells hold text and would not sum.
- **WATCH:** it hides a missing column silently. Use it only when a sheet really has no such column. Also see entry 4.5.2: `"Null"` is not the same as `"NULL"`.

---
#### 4.2.3 A column with an empty heading

**What happened:** a column that was needed had no heading in the sheet. Pandas names such columns `Unnamed: N`, where N is the column's position counting from 0 (so `Unnamed: 16` is the 17th column).

**How it was fixed:** the `columns` list used that name for the column.

```python
columns = ["Payroll Area", "Factory code", "Pers NO", "Voluntary Amount", "Tier2 Cont", "Unnamed: 16"]
```

- The name must be typed exactly like that, because it is what pandas created.
- **WATCH:** `Unnamed: N` also appears when stray cells to the right of the table hold something (a space, a note). If the table changes shape, the number changes.
- **OR:** give it a proper name after reading: `sheet_df = sheet_df.rename(columns={"Unnamed: 16": "Payroll  Month"})`.

---
#### 4.2.4 The Pers NO heading has a full stop

**What happened:** the Excel heading was `Pers NO.`, with a full stop. The code that used `"Pers NO"` failed, and leading zeros were lost.

**How it was fixed:** use the exact heading in every place: `dtype`, `columns`, the filter and `id_col`.

```python
all_sheets = pd.read_excel(path, sheet_name=None, dtype={"Pers NO.": str})
columns = ["Payroll Area ", "FACTORY CODE (Station)", "Pers NO.", "Voluntary Amount", "Tier2 Cont", "PayRoll  Month"]
selected = selected[selected["Pers NO."].notna() & (selected["Pers NO."].astype(str).str.strip() != "")]
id_col = "Pers NO."
```

- **WATCH:** the key in `dtype={...}` must match the real heading. If it does not, the option does not apply, and Pers NO is read as a number. Check with `print(df["Pers NO"].head(10))` and `df.dtypes`: text shows as `object`.
- Some of your files have the full stop and some do not. This is why `id_col` is a setting.

---
#### 4.2.5 The checker and a heading with a leading space

**What happened:** the Excel heading was `" Tier2 Cont"`, but `amount_col` in the settings was `"Tier2 Cont"`. The checking code raised a `KeyError`.

**How it was fixed:** two lines in the checking code that look the heading up by its trimmed text.

```python
out = pd.read_csv(output_file, sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
is_employee = out[id_col].str.strip() != ""
amount_col = next(c for c in columns if c.strip() == amount_col.strip())
number_cols = [c for c in [amount_col, voluntary_col] if c is not None and c in columns]
```

- `for c in columns` goes through the six headings. `c.strip() == amount_col.strip()` is `True` when they are equal after trimming the spaces at both ends.
- `(c for ... if ...)` is a **generator expression**: it hands out matching headings one at a time. `next(...)` takes the first one.
- `amount_col` now holds the real heading (with its leading space), so `out[amount_col]` works.
- `number_cols` is written again **after** the fix, because the first time it was built `amount_col` did not match any column, so the amount column was left out of the list of columns compared as numbers.
- **WATCH:** if nothing matches, `next` raises `StopIteration`. Give it a fallback: `next((c for c in columns if ...), None)`.
- **OR:** fix the setting by hand: `amount_col = " Tier2 Cont"`.

---
### 4.3 Rows

#### 4.3.1 Blank rows or a total row left inside the sheet

**What happened:** the Excel sheet had blank rows, or a total row of its own at the bottom, and the amount of that row was counted in the month total, so the total was too high.

**How it was fixed:** there were three ways to drop those rows.

```python
selected = selected[selected["Pers NO"].notna()]                                                  # (a) simplest
selected = selected[selected["Pers NO"].notna() & (selected["Pers NO"].astype(str).str.strip() != "")]   # (b) base
selected = sheet_df[columns].iloc[:-1].copy()                                                       # (c) drop the last row
dropped = selected["Pers NO"].isna().sum()                                                          # (d) count what (a) removes
print(sheet_name, "| rows dropped:", dropped)
```

- **(a)** keeps rows with a Pers NO. A cell holding only spaces counts as filled, so it stays.
- **(b)** also drops cells with only spaces. `.notna()` is `True` for a filled cell, `.astype(str).str.strip() != ""` is `True` when the text is not empty after trimming, and `&` means "and" (each side needs its own brackets).
- **(c)** `.iloc[:-1]` takes every row except the last. Use it only when the last row is always a total row. **WATCH:** on a sheet with no total row it silently removes a real employee.
- **(d)** counts the rows that (a) will remove, so you can see what is being dropped before it happens.
- **OR:** `selected.dropna(subset=["Pers NO"])` does (a) in one call.

---
#### 4.3.2 Find the row that makes the total different

**What happened:** the Python total and the Excel total differed by an amount, and the cause was not clear.

**How it was fixed:** list the Excel row numbers of the suspicious rows.

```python
gap = 0.00                                           # the difference you are hunting
df = all_sheets["SHEET NAME"]
amt = clean_amount(df["Tier2 Cont"])
has_id = df["Pers NO"].notna() & (df["Pers NO"].astype(str).str.strip() != "")
excel_row = df.index + 2

emp_rows = excel_row[has_id]
print("employee rows:", len(emp_rows), "| first:", emp_rows.min(), "| last:", emp_rows.max())
print("rows with an amount but no Pers NO:", excel_row[~has_id & (amt != 0)].tolist())
print("blank rows between employees:", excel_row[~has_id & (excel_row > emp_rows.min()) & (excel_row < emp_rows.max())].tolist())
print("a single row equal to the gap:", excel_row[amt.round(2) == gap].tolist())
print("repeated Pers NO on rows:", excel_row[has_id & df["Pers NO"].duplicated(keep=False)].tolist())
```

- `excel_row = df.index + 2` turns pandas' row numbers into Excel row numbers: pandas starts at 0 and Excel's row 1 holds the headings. **WATCH:** this is right only when the headings are on row 1 (`header=0`). With `header=2`, add 2 more.
- `excel_row[has_id]` picks the Excel row numbers of the employee rows. `.min()` and `.max()` give the first and the last.
- `~has_id & (amt != 0)` means "no Pers NO but an amount". These rows are what the filter drops; if they hold amounts, that is your gap.
- The "blank rows between employees" line shows empty rows in the middle of the data.
- The "single row equal to the gap" line finds a row whose amount equals the difference you are hunting.
- `.duplicated(keep=False)` is `True` for every row whose Pers NO appears more than once, including the first.

---
### 4.4 Numbers

#### 4.4.1 Amounts are stored as text, or hold symbols

**What happened:** some amounts were text (`"1,200.50"`, `" 1200.5"`, or with a currency sign), so a plain sum failed or left them out and the total was wrong.

**How it was fixed:** clean the text, then convert. These are the versions used:

```python
# (a) base: convert, anything unreadable becomes 0
selected["Tier2 Cont"] = pd.to_numeric(selected["Tier2 Cont"], errors="coerce").fillna(0)

# (b) delete every character that is not a digit, full stop or minus sign, then convert
selected["Tier2 Cont"] = pd.to_numeric(
    selected["Tier2 Cont"].astype(str).str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
).fillna(0)

# (c) remove commas only
selected["Tier2 Cont"] = pd.to_numeric(
    selected["Tier2 Cont"].astype(str).str.replace(",", "").str.strip(), errors="coerce"
).fillna(0)

# (d) keep Voluntary Amount as clean TEXT
selected["Voluntary Amount"] = (
    selected["Voluntary Amount"].astype(str)
    .str.replace("\xa0", "", regex=False)
    .str.strip()
    .replace("nan", "")
)

# (e) trim spaces only where the cell is text
selected["Voluntary Amount"] = selected["Voluntary Amount"].apply(lambda x: x.strip() if isinstance(x, str) else x)
```

- **(a)** `pd.to_numeric(..., errors="coerce")` turns text into numbers; what cannot be read becomes NaN, and `.fillna(0)` makes it 0. It cannot read `"1,200.50"`.
- **(b)** `.astype(str)` makes every cell text. `.str.replace(r"[^\d.-]", "", regex=True)` deletes every character that is not a digit, a full stop or a minus sign (`[^...]` means "anything except these"). **WATCH:** scientific notation such as `1e3` loses its `e`.
- **(c)** `.str.replace(",", "")` removes commas. Safer when the text only has commas, because it leaves other characters alone, so a bad cell becomes NaN and not a wrong number.
- **(d)** `"\xa0"` is a non-breaking space, an invisible space that Excel can hold. `.replace("nan", "")` turns a cell that is exactly the text "nan" into an empty cell. The result is text, which is fine for writing the file.
- **(e)** `lambda x: ...` is a one-line function. `isinstance(x, str)` is `True` for text, so numbers are left alone. `.apply(...)` runs it on every cell.
- **WATCH:** the checking code's `clean_amount` and the extraction must clean the same way, or the two totals will differ for a reason that is not an error in the data.
- **WATCH:** `.fillna(0)` hides bad cells. Use entry 4.4.2 to see how many there are.

---
#### 4.4.2 See what is wrong in an amount column

**What happened:** the amount column was cleaned, but it was not clear whether anything had been lost.

**How it was fixed:** profile the column before and after cleaning.

```python
# Which kinds of value are in the column?
print(sheet_name, sheet_df["Tier2 Cont"].map(type).value_counts().to_dict())

# Which unusual characters are in it, and how many cells are unreadable after cleaning?
import re
chars = set()
for sheet_name, sheet_df in all_sheets.items():
    if "Tier2 Cont" in sheet_df.columns:
        raw = sheet_df["Tier2 Cont"].astype(str)
        cleaned = pd.to_numeric(raw.str.replace(r"[^\d.-]", "", regex=True), errors="coerce")
        print(sheet_name, "unreadable after cleaning:", cleaned.isna().sum(), "of", len(cleaned))
        chars.update(re.findall(r"[^\d]", "".join(raw)))
print(sorted(chars))

# A full profile of one sheet:
raw = df["Tier2 Cont"]
clean = pd.to_numeric(raw.astype(str).str.replace(r"[^\d.-]", "", regex=True), errors="coerce")
has_id = df["Pers NO"].notna() & (df["Pers NO"].astype(str).str.strip() != "")
print("dtype:", raw.dtype)
print("rows:", len(df), "| kept:", has_id.sum(), "| dropped:", (~has_id).sum())
print("unreadable cells:", clean.isna().sum())
print("kept sum / all-rows sum:", round(clean[has_id].sum() / clean.sum(), 4))
print("dropped rows that hold an amount:", ((~has_id) & clean.notna() & (clean != 0)).sum())
```

- `.map(type)` replaces every cell by its Python type; `.value_counts()` counts them; `.to_dict()` makes it readable. A result such as `{float: 40, str: 3}` shows 3 text cells.
- `re.findall(r"[^\d]", text)` finds every character that is not a digit. `"".join(raw)` glues all the cells into one text. `set` removes repeats, so `sorted(chars)` shows each odd character once (commas, spaces, letters).
- "unreadable after cleaning" is how many cells became NaN, and so would be counted as 0.
- "kept sum / all-rows sum" close to 1 means almost all of the amount is on employee rows. A smaller value means the dropped rows hold amounts.
- `~has_id` means "not has_id": rows with no Pers NO.

---
#### 4.4.3 A total with decimal noise

**What happened:** a total printed as `1234.5600000000002` instead of `1234.56`.

**How it was fixed:** round it. Computers store decimals in binary, so tiny errors appear when many are added.

```python
this_month_total = round(selected["Tier2 Cont"].sum(), 2)
```

- `round(x, 2)` keeps two decimals.
- **WATCH:** never compare totals with `==`. The checking code compares with `abs(a - b) < 0.005`, "differ by less than half a cent".
- **OR:** write every number with two decimals: `to_csv(..., float_format="%.2f")`.

---
#### 4.4.4 A date written with a time in the txt

**What happened:** the Payroll Month column appeared as `2026-03-01 00:00:00` in the txt, but Excel shows `Mar-26`. The cell holds a real date; Excel only **displays** it in the format you chose.

**How it was fixed:** write the date as text in the same format as Excel shows it, in the extraction **and** on the Excel side of the checking code.

```python
# In the extraction, inside the loop, before pd.concat:
selected["Payroll  Month"] = pd.to_datetime(selected["Payroll  Month"], errors="coerce").dt.strftime("%b-%y")

# In the checking code, right after the line that builds exp:
exp["Payroll  Month"] = pd.to_datetime(exp["Payroll  Month"], errors="coerce").dt.strftime("%b-%y")
```

- `pd.to_datetime(..., errors="coerce")` makes sure the column is a real date; anything unreadable becomes blank (`NaT`) and not an error.
- `.dt.strftime("%b-%y")` rewrites each date as text: `%b` is the abbreviated month (`Mar`), `-` is a dash, `%y` is the two-digit year (`26`). Other codes: `%B` full month name, `%Y` four-digit year, `%m` month number (`03`), `%d` day.
- **WATCH:** month names are written in English with a capital first letter. If Excel shows `MAR`, add `.str.upper()` at the end.
- **WATCH:** the total rows hold `""` in this column. It is fine, because the line runs before they are added.
- The checking code needs the same line, or every row of that column is reported as different.

---
### 4.5 Totals and writing the file

#### 4.5.1 Build the total row by column name

**What happened:** the base total row is a list of values in a fixed position, so the total only lands in the right column if the order of `columns` never changes.

**How it was fixed:** build the row from a dictionary, so the total goes to the column by **name**.

```python
total_row = pd.DataFrame([{**{c: "" for c in columns}, "Tier2 Cont": this_month_total}])
grand_total_row = pd.DataFrame([{**{c: "" for c in columns}, "Tier2 Cont": grand_total}])
```

- `{c: "" for c in columns}` is a dictionary comprehension: every heading mapped to an empty text.
- `**` unpacks that dictionary into a new one. Then `"Tier2 Cont": this_month_total` is added, and replaces the blank for that key.
- `pd.DataFrame([dict])` makes a one-row table where each key is a column. The order of `columns` no longer matters.
- **OR:** the base form `pd.DataFrame([["", "", "", "", this_month_total, ""]], columns=columns)`. It is shorter, but the total's position has to be right.

---
#### 4.5.2 NULL, empty or "Null": how blanks are written

**What happened:** blank cells were written in different ways in different files, and the checking code could take employee rows for total rows, or the other way round.

**What each way does:**

```python
# The extraction:
selected = selected.replace(r"^\s*$", np.nan, regex=True)   # every blank or space-only cell becomes NaN
combined.to_csv("FactoryContribution.txt", sep="|", index=False, header=False, na_rep="NULL")   # NaN is written as NULL

selected = selected_raw.fillna("Null")                      # another way: NaN becomes the text "Null"

# The settings of the checking code:
fill_empty = "NULL"                                         # must be the same word the file uses ("" if none)

# The line in the checking code that finds employee rows:
is_employee = out[id_col].str.strip() != ""                         # only an EMPTY Pers NO means "not an employee"
is_employee = ~out[id_col].str.strip().isin(["", "NULL"])           # an empty OR NULL Pers NO means "not an employee"
```

- `na_rep` only affects true missing values (NaN). The `""` cells in the total rows stay empty, which is why the newer `is_employee` line tests for both `""` and `"NULL"`.
- **WATCH:** the older `is_employee` line fails on a file written with `na_rep="NULL"`: the NULL rows would be counted as employees.
- **WATCH:** `"Null"` and `"NULL"` are different texts, and `isin` is case-sensitive. A file written with `fillna("Null")` needs `isin(["", "Null"])`, and `fill_empty = "Null"`.
- The rule: whatever the extraction writes for blanks, `fill_empty` and `is_employee` must expect the same thing.

---
#### 4.5.3 Quick checks after combining

**What happened:** it was not certain that the combined table was complete and unchanged.

**How it was fixed:** look at the combined table.

```python
for sheet_name, sheet_df in all_sheets.items():
    print(sheet_name, sheet_df.shape)               # (rows, columns) of each sheet
print("Combined:", combined.shape)

combined.head(10)                                   # first 10 rows
combined.tail(10)                                   # last 10 rows: month total row and grand total row

total_before = sum(sheet_df.shape[0] for sheet_df in all_sheets.values())
print("Sum of individual sheets:", total_before)
print("Combined shape:", combined.shape[0])

df = combined
print(df.columns.tolist())
print(df.dtypes)                                    # object = text, float64 = numbers
print(df["Pers NO"].head(10))                       # are the leading zeros still there?
```

- `.shape` is `(rows, columns)`; `.shape[0]` is the number of rows.
- If nothing was dropped, the combined table should have **sum of sheet rows + one total row per sheet + one grand total row**. Fewer rows than that means rows were filtered out (which can be right); more means something was added twice.
- `sum(... for ... in ...)` adds up a value for every sheet in one line.
- In a notebook, only the **last** expression of a cell is shown. `combined.head(10)` followed by `combined.tail(10)` shows only the tail. Put each in its own cell, or use `display(...)`.

---
#### 4.5.4 Look at the last lines of the txt

**What happened:** it was not clear whether the txt file ended with the right total rows.

**How it was fixed:** read the file as plain text and print its last lines.

```python
with open("FactoryContribution.txt", "r", encoding="utf-8", errors="replace") as f:
    lines = f.read().splitlines()
print(lines[-2])        # the last month's total row
print(lines[-1])        # the grand total row
print(month_totals[-1]) # the same total, from Python
```

- `open(..., "r")` opens the file for reading. `with ... as f` closes it automatically afterwards.
- `encoding="utf-8"` reads the file as UTF-8. `errors="replace"` swaps any unreadable character for a `?` and does not stop.
- `.read().splitlines()` gives a list with one text per line. `[-1]` is the last line and `[-2]` the one before it.

---
### 4.6 The checking code

#### 4.6.1 A shorter checker that compares totals only

**What happened:** the full checker compares every column; sometimes only the totals mattered.

**How it was fixed:** a shorter checker. It recomputes each month from Excel, reads the txt back, and compares the totals.

```python
txt_file = "FactoryContribution.txt"
skip = []   # for example ["SHEET NAME"]

def clean_amount(series):
    return pd.to_numeric(series.astype(str).str.replace(r"[^\d.-]", "", regex=True), errors="coerce").fillna(0)

# 1. Recompute each month straight from the Excel sheets
rows = []
for sheet_name, sheet_df in all_sheets.items():
    if sheet_name in skip:
        continue
    amt = clean_amount(sheet_df["Tier2 Cont"])
    has_id = sheet_df["Pers NO"].notna() & (sheet_df["Pers NO"].astype(str).str.strip() != "")
    no_id_sum = round(amt[~has_id].sum(), 2)
    rows.append({
        "Sheet": sheet_name,
        "Employees": int(has_id.sum()),
        "Employee sum": round(amt[has_id].sum(), 2),
        "Sheet's own total row": no_id_sum if no_id_sum != 0 else None,
    })
check = pd.DataFrame(rows)

# 2. Read the txt file back and find its total rows
out = pd.read_csv(txt_file, sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
out_amt = clean_amount(out["Tier2 Cont"])
is_employee = out["Pers NO"].str.strip() != ""
is_total = ~is_employee & (out["Tier2 Cont"].str.strip() != "")
file_totals = out_amt[is_total].round(2).tolist()

# 3. Compare
if len(file_totals) != len(check) + 1:
    print("WARNING: expected", len(check) + 1, "total rows in the txt file, found", len(file_totals))
else:
    check["Total in txt"] = file_totals[:-1]
    check["Matches txt"] = (check["Employee sum"] - check["Total in txt"]).abs() < 0.005
    check["Matches sheet's own total"] = [
        "no total row" if pd.isna(own) else bool(abs(emp - own) < 0.005)
        for emp, own in zip(check["Employee sum"], check["Sheet's own total row"])
    ]
    print("Months matching the txt:", int(check["Matches txt"].sum()), "of", len(check))
    print("Grand total correct:", abs(file_totals[-1] - check["Employee sum"].sum()) < 0.005)
    print("Employee rows correct:", int(is_employee.sum()) == int(check["Employees"].sum()))

check
```

- **Step 1** builds `check`, a table with one row per sheet: the number of employees, the sum of their amounts, and the sheet's own total row if it has one (rows without a Pers NO). `rows.append({...})` adds one dictionary per sheet; `pd.DataFrame(rows)` turns the list into a table.
- **Step 2** reads the txt back. `is_total` finds rows with no Pers NO but an amount: the total rows. `file_totals` is the list of those totals, in file order.
- **Step 3** warns if the number of total rows is not one per sheet plus one grand total. Otherwise it adds the txt total next to each month. `file_totals[:-1]` is every total **except the last**, which is the grand total.
- The list comprehension `[... for emp, own in zip(a, b)]` walks two columns side by side. `zip` pairs them. `pd.isna(own)` is `True` when the sheet has no total row of its own.
- **Difference from the base checker:** it does not compare every column of every row, only totals and the employee count.

---
#### 4.6.2 Pick the newest txt file automatically

**What happened:** the checker had to be edited each time for the file name.

**How it was fixed:** take the most recently saved txt file in the folder.

```python
import os

txt_file = max((f for f in os.listdir() if f.endswith(".txt")), key=os.path.getmtime)
print(txt_file)
```

- `os.listdir()` lists the file names in the current folder. `f.endswith(".txt")` keeps the txt files.
- `max(..., key=os.path.getmtime)` returns the file with the largest "last modified" time, which is the newest.
- **WATCH:** it picks the newest txt, not the right one. A master file written after the contribution file would be picked instead. `print(txt_file)` is there so you can see it.

---
#### 4.6.3 Quiet the warnings and show more

```python
import warnings
warnings.filterwarnings("ignore")              # stop printing warnings (errors still stop the code)

pd.set_option("display.max_columns", None)     # show all columns, with no "..." in the middle
pd.set_option("display.max_colwidth", None)    # show long text cells in full
display(sheet_df.head())                       # show a table from anywhere in a cell
```

- A warning is not an error. For example, nbformat's `DuplicateCellId` appears when notebooks are merged, and it is fixed automatically.
- `display(...)` shows a table even if it is not the last line of the cell.

---
### 4.7 General checks, error meanings and habits

#### 4.7.1 Common errors and what they mean

| Error | What it means | What to do |
|---|---|---|
| `NameError: name 'x' is not defined` | The variable does not exist yet. | Run the cell that creates it first. After a kernel restart, every earlier cell must be run again. |
| `KeyError: 'Some heading'` | A column name does not match the Excel heading exactly. | Print `df.columns.tolist()` and copy the name (entries 4.2.1 to 4.2.5). |
| `TypeError: unsupported operand type(s) for +: 'int' and 'str'` | Text was mixed with numbers in a sum. | Convert the column to numbers first (entry 4.4.1). |
| `ValueError: 5 columns passed, passed data had 6 columns` | A row you built has more values than there are headings. | Build the row from the `columns` list, by name (entry 4.5.1). |
| `FileNotFoundError` | The path or the file name is wrong. | Check the spelling, a doubled `.xlsx.xlsx`, and the folder (entry 4.7.2). |
| `ValueError: Worksheet named 'X' not found` | The sheet name is not exact. | Copy the name from `list(all_sheets.keys())` or `pd.ExcelFile(path).sheet_names`. |
| `ImportError` or `ModuleNotFoundError` (`xlrd`, `openpyxl`, `nbformat`) | A library is not installed. | `%pip install ...`, then restart the kernel (entry 4.1.1). |
| `IndentationError`, or "unindent does not match any outer indentation level" | The lines of one block do not start with the same number of spaces. | Use 4 spaces for every level. One notebook used 3 spaces and worked only because every line in the block used 3. |

---
#### 4.7.2 Find the folder, the file, and where the txt is saved

**What happened:** `FileNotFoundError`, or a txt file that could not be found after writing it.

**How it was fixed:** ask Python what it can see.

```python
import os

folder = r"C:\path\to\folder"
print(os.path.exists(folder))                       # True if Python can find the folder
if os.path.exists(folder):
    print([repr(f) for f in os.listdir(folder)])    # every file name, with hidden spaces visible

print(os.getcwd())                                  # the folder where a txt written with a bare name is saved
print("Saved:", os.path.abspath("FactoryContribution.txt"))   # the full path of the file, after writing it
```

- `import os` loads Python's built-in tools for files and folders.
- `os.path.exists(folder)` is `True` or `False`. It is the fastest way to find out whether the path is the problem.
- `os.listdir(folder)` lists the names in the folder. `[repr(f) for f in ...]` is a list comprehension that shows each name with its quotation marks, so a trailing space or a doubled `.xlsx.xlsx` is visible.
- `os.getcwd()` is the folder the notebook is running in. A txt written with only a name (no folder) is saved there.
- `os.path.abspath(name)` turns a bare name into its full path, so you know exactly where the file went.
- **WATCH:** in VS Code you can right-click a file and choose **Copy Path**, so you never type a path by hand.

---
#### 4.7.3 Good habits

- Python is case-sensitive and indentation matters, so copy names from printed output instead of typing them.
- In a working notebook, restart the kernel and use Run All, so old variables do not remain.
- Keep only one loop in the cell. A second loop doubles every month.
- Every cell that uses `pd` or `np` needs its own import line if you want it to run alone.
- Compare each printed total and row count with Excel before you trust the file.
- Keep an untouched copy of every original payroll file.
- Share only counts, types and headings, never payroll values.

## 5. How the base grew

The base extraction started simple, and lines were added as problems appeared. This is what each line is for.

| Line | The problem it solves |
|---|---|
| `dtype={"Pers NO": str}` | Pers NO lost its leading zeros when read as a number. |
| `.copy()` after `sheet_df[columns]` | Avoids the `SettingWithCopyWarning` and keeps the original sheet unchanged. |
| `selected[selected["Pers NO"].notna() & (... != "")]` | Removes blank rows and any total row already in the sheet, so nothing is counted twice. |
| `selected.replace(r"^\s*$", np.nan, regex=True)` | Makes every blank or space-only cell the same kind of missing value. |
| `pd.to_numeric(..., errors="coerce").fillna(0)` | Amounts stored as text made the sum wrong. |
| `print(sheet_name, this_month_total, len(selected))` and `.tail(2)` | Lets you compare each month with Excel's own total before the file is written. |
| `na_rep="NULL"` in `to_csv` | Missing values are written as the word NULL. |
| `grand_total = sum(month_totals)` | Adds the monthly totals. Summing the Tier2 column of `combined` would count the total rows as well. |
| `round(..., 2)` | Hides decimal noise such as `1234.5600000000002`. |
| `abs(a - b) < 0.005` in the checker | Compares decimal numbers safely, without `==`. |
| `~...isin(["", "NULL"])` in the checker | Treats both empty and NULL as "no Pers NO". |
| `dt.strftime("%b-%y")` | Writes the payroll month the way Excel shows it. |